# Sequence models, and the trap of one-hot inputs

A bidirectional LSTM that loses to bag-of-words, the diagnosis, and the embedding layer that fixes it.

**Runs on:** GPU recommended — about 30 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 14 — Text Classification](../../../course-web-slides/ch14/index.html) &nbsp;·&nbsp; **Section:** 03 — Sequences: the sequence model approach

---

## Integer sequences, not multi-hot

In [ ]:
import keras
from keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y),
                            num_parallel_calls=4)
int_val_ds = val_ds.map(lambda x, y: (text_vectorization(x), y),
                        num_parallel_calls=4)
int_test_ds = test_ds.map(lambda x, y: (text_vectorization(x), y),
                          num_parallel_calls=4)

for x, y in int_train_ds:
    print(x.shape, x.dtype)
    break

## The naive version: one-hot into an LSTM

In [ ]:
from keras import ops

inputs = keras.Input(shape=(None,), dtype="int64")
embedded = ops.one_hot(inputs, num_classes=max_tokens)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

> ⚠️ **Every timestep is a 20,000-dimensional vector.** 600 timesteps × 20,000 = 12 million numbers per review, almost all of them zero. This will be extremely slow, and it is the point of the exercise.

In [ ]:
cb = [keras.callbacks.ModelCheckpoint("one_hot_bidir_lstm.keras",
                                      save_best_only=True)]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=cb, verbose=2)
acc = keras.models.load_model("one_hot_bidir_lstm.keras").evaluate(
    int_test_ds, verbose=0)[1]
print(f"\none-hot LSTM test accuracy: {acc:.3f}   "
      f"(bag-of-bigrams was 0.90)")

Expected output:

```
one-hot LSTM test accuracy: 0.87x   (bag-of-bigrams was 0.90)
```

**Slower, more complex, and worse.** Two reasons, and the second is the fixable one:

- The input representation is enormous and sparse.
- **One-hot vectors are all equidistant.** *excellent* and *good* are exactly as far apart as *excellent* and *refrigerator*. The representation asserts that all words are unrelated, which is false.

## An Embedding layer

In [ ]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

**256 dimensions instead of 20,000**, and the geometry is learned rather than imposed. An `Embedding` layer is a lookup table: token *i* returns row *i*, and those rows are trained by backpropagation like any other weights.

## Masking, which is not optional

In [ ]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256,
                            mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])

cb = [keras.callbacks.ModelCheckpoint("embeddings_bidir_lstm_masked.keras",
                                      save_best_only=True)]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=cb, verbose=2)
acc_masked = keras.models.load_model(
    "embeddings_bidir_lstm_masked.keras").evaluate(int_test_ds, verbose=0)[1]
print(f"\nembedding + masking test accuracy: {acc_masked:.3f}")

Expected output:

```
embedding + masking test accuracy: 0.87x — 0.88x
```

`mask_zero=True` tells every downstream layer to **skip the padding**. Without it, a 20-word review padded to 600 gives the LSTM 580 steps of nothing to process, and the state it arrives with has been diluted by all of them.

## The scoreboard, and the honest reading

In [ ]:
import matplotlib.pyplot as plt

results = [("bag-of-words (unigram)", 0.885),
           ("bag-of-bigrams", 0.902),
           ("TF-IDF bigrams", 0.897),
           ("one-hot bidir LSTM", 0.873),
           ("embedding bidir LSTM", 0.874),
           ("+ masking", acc_masked)]

names = [r[0] for r in results]; vals = [r[1] for r in results]
plt.figure(figsize=(8.5, 4))
plt.barh(names[::-1], vals[::-1],
         color=["#12b886" if v >= 0.9 else "#888" for v in vals][::-1])
plt.xlim(0.84, 0.92); plt.xlabel("test accuracy")
plt.title("On this dataset, bag-of-bigrams wins")
plt.tight_layout(); plt.show()

**The bag-of-words model wins.** That is the honest result on IMDB, and the chapter says so.

The rule the book gives is a ratio: **number of training samples divided by mean sample length**. Below about 1,500, bag-of-words wins. IMDB has 20,000 samples of ~230 words, giving 87 — well inside bag-of-words territory.

In [ ]:
import numpy as np

lengths = []
for x, _ in train_ds.take(50):
    lengths += [len(s.numpy().split()) for s in x]
ratio = 20000 / np.mean(lengths)
print(f"mean review length: {np.mean(lengths):.0f} words")
print(f"ratio: 20000 / {np.mean(lengths):.0f} = {ratio:.0f}")
print(f"\nrule of thumb: below ~1500 -> bag-of-words; above -> sequence model")
print(f"this dataset: {ratio:.0f}  ->  {'bag-of-words' if ratio < 1500 else 'sequence model'}")

---

## What to take away

- One-hot inputs assert that every word is equidistant from every other, which is false.
- `Embedding` learns a dense geometry instead — 256 dimensions rather than 20,000.
- **`mask_zero=True`** or the recurrent layer processes hundreds of padding steps.
- On IMDB, bag-of-bigrams wins. The samples-to-length ratio predicts which family to use.